In [5]:
import pandas as pd
import random
from datetime import datetime, timedelta

# ==========================================
# HEATSHIELD AI - THERMAL STRESS DATASET
# Generate 10,000 Synthetic Weather Records
# ==========================================

NUM_RECORDS = 10000

# Reproducible results
random.seed(42)

# ==========================================
# INDIAN CITIES WITH APPROXIMATE COORDINATES
# ==========================================

locations = {
    "Hyderabad": (17.3850, 78.4867),
    "Delhi": (28.6139, 77.2090),
    "Chennai": (13.0827, 80.2707),
    "Mumbai": (19.0760, 72.8777),
    "Ahmedabad": (23.0225, 72.5714),
    "Jaipur": (26.9124, 75.7873),
    "Vijayawada": (16.5062, 80.6480),
    "Madanapalle": (13.5500, 78.5000),
    "Khammam": (17.2473, 80.1514)
}


# ==========================================
# CITY CLIMATE PROFILES
# ==========================================

city_profiles = {
    "Hyderabad": {"temp": (25, 47), "humidity": (20, 70)},
    "Delhi": {"temp": (22, 49), "humidity": (15, 75)},
    "Chennai": {"temp": (26, 42), "humidity": (50, 90)},
    "Mumbai": {"temp": (25, 39), "humidity": (55, 95)},
    "Ahmedabad": {"temp": (24, 48), "humidity": (15, 70)},
    "Jaipur": {"temp": (22, 48), "humidity": (10, 65)},
    "Vijayawada": {"temp": (25, 46), "humidity": (30, 85)},
    "Madanapalle": {"temp": (18, 38), "humidity": (30, 80)},
    "Khammam": {"temp": (24, 47), "humidity": (25, 80)}
}


# ==========================================
# HEAT INDEX CALCULATION
# ==========================================

def calculate_heat_index(temp_c, humidity):

    # Heat index is mainly meaningful
    # at higher temperatures
    if temp_c < 27:
        return temp_c

    # Convert Celsius to Fahrenheit
    temp_f = temp_c * 9 / 5 + 32

    # Rothfusz Regression Formula
    hi_f = (
        -42.379
        + 2.04901523 * temp_f
        + 10.14333127 * humidity
        - 0.22475541 * temp_f * humidity
        - 0.00683783 * temp_f * temp_f
        - 0.05481717 * humidity * humidity
        + 0.00122874 * temp_f * temp_f * humidity
        + 0.00085282 * temp_f * humidity * humidity
        - 0.00000199 * temp_f * temp_f * humidity * humidity
    )

    # Convert back to Celsius
    hi_c = (hi_f - 32) * 5 / 9

    # Heat index should not be below air temperature
    return max(temp_c, hi_c)


# ==========================================
# THERMAL STRESS SCORE
# ==========================================

def calculate_thermal_stress(
    temp,
    humidity,
    wind,
    solar_radiation
):

    # Temperature contribution (0-40)
    temp_score = ((temp - 20) / 30) * 40
    temp_score = max(0, min(40, temp_score))

    # Humidity contribution (0-20)
    humidity_score = (humidity / 100) * 20

    # Low wind increases heat stress (0-15)
    wind_score = max(0, (5 - wind) / 5) * 15
    wind_score = min(15, wind_score)

    # Solar radiation contribution (0-25)
    solar_score = (solar_radiation / 1100) * 25
    solar_score = max(0, min(25, solar_score))

    # Total score
    score = (
        temp_score
        + humidity_score
        + wind_score
        + solar_score
    )

    # Natural environmental variation
    score += random.uniform(-3, 3)

    # Keep between 0 and 100
    score = max(0, min(100, score))

    return round(score, 2)


# ==========================================
# RISK LEVEL CLASSIFICATION
# ==========================================

def get_risk_level(score):

    if score < 25:
        return "Low"

    elif score < 50:
        return "Moderate"

    elif score < 75:
        return "High"

    else:
        return "Extreme"


# ==========================================
# GENERATE DATA
# ==========================================

data = []

# Heat season period
start_date = datetime(2025, 3, 1)

for i in range(NUM_RECORDS):

    # Select random city
    city = random.choice(list(locations.keys()))

    latitude, longitude = locations[city]

    # City climate profile
    profile = city_profiles[city]

    # Random date during March-June
    random_days = random.randint(0, 121)

    date_time = (
        start_date
        + timedelta(
            days=random_days,
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59)
        )
    )

    hour = date_time.hour
    month = date_time.month

    # ======================================
    # TEMPERATURE GENERATION
    # ======================================

    min_temp, max_temp = profile["temp"]

    # Seasonal variation
    month_adjustment = {
        3: -3,
        4: 0,
        5: 3,
        6: 1
    }

    seasonal_adjustment = month_adjustment.get(
        month,
        0
    )

    # Temperature based on time of day

    if 11 <= hour <= 16:

        temp = random.uniform(
            max_temp - 6,
            max_temp
        )

    elif (
        7 <= hour <= 10
        or 17 <= hour <= 19
    ):

        temp = random.uniform(
            min_temp + 5,
            max_temp - 8
        )

    else:

        temp = random.uniform(
            min_temp,
            min_temp + 8
        )

    temp += seasonal_adjustment

    # Keep temperature within realistic range
    temp = max(
        min_temp - 3,
        min(max_temp + 2, temp)
    )

    temp = round(temp, 2)


    # ======================================
    # HUMIDITY GENERATION
    # ======================================

    min_humidity, max_humidity = profile["humidity"]

    # Higher temperature generally means lower humidity
    temperature_factor = (
        (temp - min_temp)
        / (max_temp - min_temp)
    )

    humidity = (
        max_humidity
        - temperature_factor
        * (max_humidity - min_humidity)
        + random.uniform(-8, 8)
    )

    # Night time usually has higher humidity
    if hour < 7 or hour > 18:
        humidity += random.uniform(5, 15)

    humidity = max(
        min_humidity,
        min(max_humidity, humidity)
    )

    humidity = round(humidity, 2)


    # ======================================
    # WIND SPEED
    # ======================================

    wind_speed = random.uniform(
        0.2,
        8.0
    )

    # Dangerous low-wind conditions
    if random.random() < 0.25:

        wind_speed = random.uniform(
            0.1,
            2.0
        )

    wind_speed = round(
        wind_speed,
        2
    )


    # ======================================
    # SOLAR RADIATION
    # ======================================

    if 6 <= hour <= 18:

        # Peak around 12 PM
        peak_factor = (
            1
            - abs(hour - 12) / 6
        )

        solar_radiation = (
            peak_factor * 1050
            + random.uniform(-80, 80)
        )

        solar_radiation = max(
            0,
            solar_radiation
        )

    else:

        # Night time
        solar_radiation = random.uniform(
            0,
            5
        )

    solar_radiation = min(
        1100,
        solar_radiation
    )

    solar_radiation = round(
        solar_radiation,
        2
    )


    # ======================================
    # HEAT INDEX
    # ======================================

    heat_index = calculate_heat_index(
        temp,
        humidity
    )

    heat_index = round(
        heat_index,
        2
    )


    # ======================================
    # THERMAL STRESS SCORE
    # ======================================

    thermal_stress_score = (
        calculate_thermal_stress(
            temp,
            humidity,
            wind_speed,
            solar_radiation
        )
    )


    # ======================================
    # RISK LEVEL
    # ======================================

    risk_level = get_risk_level(
        thermal_stress_score
    )


    # ======================================
    # SAVE RECORD
    # ======================================

    data.append({

        "record_id": i + 1,

        "date": date_time.strftime(
            "%Y-%m-%d"
        ),

        "time": date_time.strftime(
            "%H:%M"
        ),

        "location": city,

        "latitude": latitude,

        "longitude": longitude,

        "temperature_c": temp,

        "humidity_percent": humidity,

        "wind_speed_ms": wind_speed,

        "solar_radiation_wm2":
            solar_radiation,

        "heat_index_c":
            heat_index,

        "thermal_stress_score":
            thermal_stress_score,

        "thermal_stress_level":
            risk_level
    })


# ==========================================
# CREATE DATAFRAME
# ==========================================

df = pd.DataFrame(data)


# ==========================================
# SAVE CSV FILE
# ==========================================

file_name = (
    "HeatShield_AI_Thermal_Stress_Dataset.csv"
)

df.to_csv(
    file_name,
    index=False
)


# ==========================================
# DISPLAY RESULTS
# ==========================================

print("=" * 60)

print(
    "HEATSHIELD AI DATASET GENERATED SUCCESSFULLY!"
)

print("=" * 60)

print(
    "\nTotal Records:",
    len(df)
)

print("\nFirst 5 Records:")

print(
    df.head()
)

print("\nDataset Information:")

df.info()


print(
    "\nThermal Stress Distribution:"
)

print(
    df[
        "thermal_stress_level"
    ].value_counts()
)


print("\nCSV File Saved As:")

print(
    file_name
)


print(
    "\nDataset Summary:"
)

print(
    df[
        [
            "temperature_c",
            "humidity_percent",
            "wind_speed_ms",
            "solar_radiation_wm2",
            "heat_index_c",
            "thermal_stress_score"
        ]
    ].describe()
)

HEATSHIELD AI DATASET GENERATED SUCCESSFULLY!

Total Records: 10000

First 5 Records:
   record_id        date   time   location  latitude  longitude  \
0          1  2025-03-04  23:17      Delhi   28.6139    77.2090   
1          2  2025-03-28  07:32      Delhi   28.6139    77.2090   
2          3  2025-06-12  00:48  Ahmedabad   23.0225    72.5714   
3          4  2025-04-15  11:38      Delhi   28.6139    77.2090   
4          5  2025-05-13  06:45     Jaipur   26.9124    75.7873   

   temperature_c  humidity_percent  wind_speed_ms  solar_radiation_wm2  \
0          20.96              75.0           5.98                 2.95   
1          32.43              52.8           5.78               162.12   
2          31.45              64.5           1.41                 1.68   
3          44.59              17.5           1.85               807.61   
4          25.56              65.0           7.88                58.64   

   heat_index_c  thermal_stress_score thermal_stress_level  
0    